# Using the OpenAI Responses API with Strands Agents

## Overview

OpenAI's GPT-5 models are callable through the [OpenAI Responses API](https://strandsagents.com/docs/user-guide/concepts/model-providers/openai-responses/). Strands' `OpenAIResponsesModel` provider talks to that API, so you can run an agent against an OpenAI model using your OpenAI API key.

In this example we use `gpt-5-mini` as the agent's model, with a simple `current_time` and `current_weather` tool use case. The API key is read from the `OPENAI_API_KEY` environment variable.

## Tutorial Details

| Information            | Details                                        |
|:-----------------------|:-----------------------------------------------|
| Agent structure        | Single agent                                   |
| Model                  | `gpt-5-mini` (from OpenAI)                     |
| Runs on                | OpenAI API                                     |
| Strands model provider | `OpenAIResponsesModel`                          |
| Custom tools           | current_time, current_weather                  |

## Architecture

<div style="text-align:center">
    <img src="images/simple_agent.png" width="65%" />
</div>

## What you'll learn
* Configure an OpenAI model with the `OpenAIResponsesModel` provider
* Build a tool-using agent and run it
* Inspect the response metrics

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* Set `OPENAI_API_KEY` in your environment (optionally `OPENAI_MODEL_ID`, default `gpt-5-mini`)

Install the required packages for our agent:

In [ ]:
# installing pre-requisites
!pip install -r requirements.txt

### Importing dependency packages

Now let's import the dependency packages:

In [ ]:
import os
from datetime import datetime
from datetime import timezone as tz
from typing import Any
from zoneinfo import ZoneInfo

from strands import Agent, tool
from strands.models.openai_responses import OpenAIResponsesModel

### Authenticate to OpenAI

The provider authenticates with your OpenAI API key. This notebook reads it from the `OPENAI_API_KEY` environment variable, so make sure it is set before you start Jupyter. Let's confirm it is available (without printing it):

In [ ]:
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your environment"

### Define the tools

Let's set up two tools for the agent to call:

In [ ]:
@tool
def current_time(timezone: str = "UTC") -> str:
    if timezone.upper() == "UTC":
        timezone_obj: Any = tz.utc
    else:
        timezone_obj = ZoneInfo(timezone)

    return datetime.now(timezone_obj).isoformat()


@tool
def current_weather(city: str) -> str:
    # Dummy implementation. Replace with actual weather API call.
    return "sunny"

### Build the model and agent

Now create the model, passing the API key through `client_args`. Then build the agent with the tools.

In [ ]:
model = OpenAIResponsesModel(
    model_id=os.getenv("OPENAI_MODEL_ID", "gpt-5-mini"),
    client_args={"api_key": os.environ["OPENAI_API_KEY"]},
)

system_prompt = "You are a simple agent that can tell the time and the weather"
agent = Agent(model=model, system_prompt=system_prompt, tools=[current_time, current_weather])

### Run the agent

Let's ask a simple question first. The model answers directly, without needing a tool:

In [ ]:
response = agent("What is the capital of British Columbia?")

Now ask something that needs the tools. The agent calls `current_time` and `current_weather` and uses their results:

In [ ]:
response = agent("What is the time and weather in Philadelphia?")

### Inspect the response

Let's look at the usage for the last query by examining the response `metrics`:

In [ ]:
from pprint import pprint

pprint(vars(response.metrics))

## Summary

In this notebook you called an OpenAI model with the `OpenAIResponsesModel` provider and the Responses API, built a tool-using agent, ran it, and inspected its metrics. This completes the model providers tutorial: you have now run a Strands agent against a local Ollama model, an OpenAI model through LiteLLM, and an OpenAI model through the Responses API.